# Rutendo AI — TFLite Export for Pruned No-NMS Model
Upload `yolo11n.pt` (original model, 5.4 MB) then run all cells.

The notebook prunes from 80→11 COCO classes and exports to TFLite (FP32 + INT8).

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
from google.colab import files
import torch
import torch.nn as nn
import os
import shutil
from pathlib import Path

### Step 1: Upload yolo11n.pt

In [ ]:
uploaded = files.upload()
model_path = list(uploaded.keys())[0]
print(f'Using: {model_path}')

### Step 2: Prune detection head from 80 → 11 classes
Keeps: person, bicycle, car, motorcycle, bus, truck, traffic light, stop sign, cat, dog, chair

COCO indices: 0,1,2,3,5,7,9,11,15,16,56

In [ ]:
KEEP = [0, 1, 2, 3, 5, 7, 9, 11, 15, 16, 56]
NEW_NAMES = ["person","bicycle","car","motorcycle","bus","truck","traffic light","stop sign","cat","dog","chair"]
NUM_KEEP = len(KEEP)

model = YOLO(model_path)
m = model.model
detect = m.model[-1]
print(f'Original nc: {detect.nc}')

# Prune cv3 class prediction layers
new_cv3 = nn.ModuleList()
for i, seq in enumerate(detect.cv3):
    layers = list(seq)
    last_conv_idx = None
    for j in range(len(layers)-1, -1, -1):
        if isinstance(layers[j], nn.Conv2d):
            last_conv_idx = j
            break
    if last_conv_idx is None:
        new_cv3.append(seq)
        continue
    old_conv = layers[last_conv_idx]
    new_conv = nn.Conv2d(old_conv.in_channels, NUM_KEEP, old_conv.kernel_size[0],
                         stride=old_conv.stride[0], padding=old_conv.padding[0],
                         bias=old_conv.bias is not None)
    with torch.no_grad():
        new_conv.weight.data = old_conv.weight.data[KEEP, :, :, :].clone()
        if old_conv.bias is not None:
            new_conv.bias.data = old_conv.bias.data[KEEP].clone()
    layers[last_conv_idx] = new_conv
    new_cv3.append(nn.Sequential(*layers))

detect.cv3 = new_cv3
detect.nc = NUM_KEEP
detect.no = NUM_KEEP + detect.reg_max * 4
m.names = {i: NEW_NAMES[i] for i in range(NUM_KEEP)}
print(f'Pruned nc: {detect.nc}')
for i, seq in enumerate(detect.cv3):
    last = list(seq)[-1]
    print(f'  cv3[{i}] out_channels={last.out_channels}')

### Step 3: Export to TFLite (no NMS)

In [ ]:
# FP32 TFLite (no quantization, good for GPU delegate)
out_fp32 = model.export(format='tflite', imgsz=640, nms=False)
print(f'FP32: {out_fp32}')
fp32_size = os.path.getsize(out_fp32) / 1024 / 1024
print(f'  Size: {fp32_size:.1f} MB')

In [ ]:
# INT8 quantized TFLite (smaller, faster on NPU)
out_int8 = model.export(format='tflite', imgsz=640, nms=False, int8=True)
print(f'INT8: {out_int8}')
int8_size = os.path.getsize(out_int8) / 1024 / 1024
print(f'  Size: {int8_size:.1f} MB')

### Step 4: Download results

In [ ]:
for f in os.listdir():
    if f.endswith('.tflite'):
        files.download(f)
        print(f'Downloaded: {f}')

# Also save labels
with open('labels.txt', 'w') as f:
    f.write('\n'.join(NEW_NAMES) + '\n')
files.download('labels.txt')
print('Downloaded: labels.txt')